In [1]:
from graph_build import create_dummy_knowledge_graph
from graph_retriver import GraphRetriver

c:\Users\migue\Documents\Code_projects\Python\envs\AI_general_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 621.07it/s, Materializing param=pooler.dense.weight]                             


In [29]:
from collections import deque
kg = create_dummy_knowledge_graph()
graph_retrive = GraphRetriver(kg)
query = 'What was the relation between Hitler and Albert Einstein?'
relevant_nodes = graph_retrive.get_relevant_nodes(query)
print(relevant_nodes)
triples = set()
visited = set()
queue = deque()

for node in relevant_nodes:
    queue.append((node,0))
    visited.add(node)

['Hitler', 'Albert Einstein']


In [30]:
while queue:
    print(queue)
    current_node, depth = queue.popleft() #list of the pending 'visits'
    if depth >= 2:
        continue

    for neighbor in kg.G.neighbors(current_node):
        (s, r, o) = (current_node,
                     kg.G[current_node][neighbor]['relation'],
                     neighbor)
        print(s,r,o)




deque([('Hitler', 0), ('Albert Einstein', 0)])
Hitler born in Braunau am Inn, Austria
Hitler perscuted Jews
deque([('Albert Einstein', 0)])
Albert Einstein developed Theory of Relativity
Albert Einstein born in Ulm, Germany
Albert Einstein died in Princeton, USA
Albert Einstein was Jews


In [31]:
relevant_triplets = graph_retrive.retrieve(query)
for triple in relevant_triplets:
    print(triple)

('Hitler', 'born in', 'Braunau am Inn, Austria')
('Hitler', 'perscuted', 'Jews')
('Albert Einstein', 'developed', 'Theory of Relativity')
('Albert Einstein', 'born in', 'Ulm, Germany')
('Albert Einstein', 'died in', 'Princeton, USA')
('Albert Einstein', 'was', 'Jews')


In [23]:
model_instructions = '''You are a question answering system.
You must answer using ONLY the provided graph facts.
If the facts do not contain the answer, reply:
"Not enough information."
Graph Facts:
'''

# Using Smolagent

In [102]:
from smolagents import LiteLLMModel, ChatMessage
#Define the model to answer the query with the context:
# Initialize the local Ollama model
model = LiteLLMModel(
    model_id="ollama_chat/qwen2.5:1.5b-Instruct", 
    api_base="http://localhost:11434", # Default Ollama port
    num_ctx=8192 # Context window
)
messages = [
            ChatMessage(role="system", content=[{"type": "text", "text": model_instructions+context}]),
            ChatMessage(role="user", content=[{"type": "text", "text": query}])
        ]
answer = model.generate(messages=messages)
print(answer.content)

Hitler was a Nazi who persecuted Jews, including Albert Einstein.


# Using Ollama model

In [103]:
import ollama
stream = ollama.chat(
    model='qwen2.5:1.5b-Instruct',
    messages=[{'role': 'system', 'content': model_instructions + context},
              {'role': 'user', 'content': query}],
    stream=True,
)
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Not enough information.

# Using HF inference model

In [105]:
from dotenv import load_dotenv
from smolagents import InferenceClientModel
import os
load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN")
model_for_extraction = InferenceClientModel(model_id = 'Qwen/Qwen2.5-Coder-7B-Instruct') 
messages=[{'role': 'system', 'content': model_instructions + context},
                        {'role': 'user', 'content': query}]
model_for_extraction(messages).content

'Hitler persecuted Jews.  \nAlbert Einstein developed the Theory of Relativity.  \nAlbert Einstein was Jewish.'